# SCALPEL annotation pipeline: test

Runs the new `cell_type_annotation.py` steps in memory (reuses the MapMyCells JSON, writes
only to `cell_type_annotation/_scalpel_check/`):

1. MapMyCells output
2. SCALPEL QC: DoubleMAD per supertype, with bimodal handling (`scalpel_qc`)
3. `group_cell_types()`
4. Leiden, majority label per cluster, QC-failed cells voting `Undefined` → `cell_type_revised`

Compared with the same vote at Leiden res 20, and with **pseudobulk MapMyCells**: mean log2(1 + CPM) of QC-passed cells per res-20 cluster (the same centroid definition as the ABC reference), mapped once per cluster; a cluster is `Undefined` if most of its cells failed QC.

Needs the `scalpel-annotation` branch pulled in `REPO`.

In [ ]:
# ruff: noqa
import os, sys, json, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.sparse as sp

REPO = str(Path.cwd().parents[1])   # this notebook lives in REPO/notebooks/script_setup
assert Path(REPO, "cellseg_benchmark").is_dir(), f"{REPO} is not the repo root"
sys.path.insert(0, REPO)
for m in [m for m in sys.modules if m.startswith("cellseg_benchmark")]:
    del sys.modules[m]
import cellseg_benchmark.cell_annotation_utils as anno_utils
from cellseg_benchmark._constants import cell_type_colors
print(anno_utils.__file__)
assert hasattr(anno_utils, "scalpel_qc"), f"{anno_utils.__file__} has no scalpel_qc: run `git log -1` in {REPO}, expected the scalpel-annotation commit"

warnings.filterwarnings("ignore")
logger = logging.getLogger("scalpel_check")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
pd.set_option("display.width", 160)

sample_name = "htra1_s3_r0"
seg_method = "Proseg_3D_Cellpose_1_nuclei_model"
data_dir = "/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark"
mad_factor, leiden_res = 3.0, 10.0

method_path = Path(data_dir, "samples", sample_name, "results", seg_method)
annotation_path = method_path / "cell_type_annotation"
out_path = annotation_path / "_scalpel_check"
out_path.mkdir(parents=True, exist_ok=True)

## 1. SCALPEL QC

In [ ]:
json_path = max(annotation_path.glob(f"mapmycells_out/*MapMyCells_{sample_name}_{seg_method}.json"),
                key=os.path.getmtime)
with open(json_path, "rb") as src:
    mmc = anno_utils.process_mapmycells_output(json.load(src))

qc = anno_utils.scalpel_qc(mmc["allen_cor_SUPT"], mmc["allen_SUPT"], mad_factor)
mmc = mmc.join(qc)

n_per_st = mmc.groupby("allen_SUPT").size()
print(json_path.name)
print(f"cells: {len(mmc):,}   supertypes: {len(n_per_st)}   median cells/supertype: {n_per_st.median():.0f}")
print(f"cells in supertypes with < 10 cells: {n_per_st[n_per_st < 10].sum() / len(mmc):.1%}")
print(f"cor_SUPT identical to cor_CLUS: {(mmc.allen_cor_SUPT == mmc.allen_cor_CLUS).mean():.1%} of cells "
      "(near 100% means SUPT was dropped in MapMyCells and inherits the cluster correlation)")
print(f"bimodal supertypes: {mmc.loc[mmc.is_bimodal_supertype, 'allen_SUPT'].nunique()} "
      f"({mmc.is_bimodal_supertype.mean():.1%} of cells)")
print(f"QC failed: {(~mmc.qc_passed).mean():.2%} of cells")

In [ ]:
# Optional: check the Python port of LaplacesDemon::is.bimodal against R, if rpy2 is available
try:
    import rpy2.robjects as ro
    ro.r("library(LaplacesDemon)")
    rows = []
    for st, x in mmc.loc[mmc.allen_cor_SUPT > 0].groupby("allen_SUPT").allen_cor_SUPT:
        v = x.to_numpy()
        if len(v) < 2 or np.ptp(v) == 0:
            continue
        r = sorted(ro.r["Modes"](ro.FloatVector(v))[0])
        rows.append({"supertype": st, "R": len(r) == 2, "python": len(anno_utils._modes(v)[0]) == 2})
    cmp = pd.DataFrame(rows)
    print(f"is.bimodal agrees with R for {(cmp.R == cmp.python).mean():.1%} of {len(cmp)} supertypes")
    display(cmp[cmp.R != cmp.python])
except Exception as e:
    print("R check skipped:", e)

In [ ]:
bimodal = mmc.loc[mmc.is_bimodal_supertype, "allen_SUPT"].value_counts().index[:8].tolist()
largest = mmc.loc[~mmc.is_bimodal_supertype, "allen_SUPT"].value_counts().index[:4].tolist()
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, st in zip(axes.flat, bimodal + largest):
    sub = mmc[mmc.allen_SUPT == st]
    ax.hist(sub.allen_cor_SUPT, bins=50, color="C1" if st in bimodal else "C0")
    ax.axvline(sub.qc_thr.max(), color="k", ls="--")
    ax.set_title(f"{st[:35]}\nn={len(sub)}, failed {(~sub.qc_passed).mean():.0%}", fontsize=8)
for ax in axes.flat[len(bimodal + largest):]:
    ax.axis("off")
plt.suptitle("orange: bimodal supertypes, blue: largest unimodal; dashed: threshold")
plt.tight_layout()
plt.savefig(out_path / "1_qc_thresholds.png", dpi=120)
plt.show()

## 2. Pipeline, as in the script

In [ ]:
from spatialdata import read_zarr

mmc["allen_SUBC"] = anno_utils.group_cell_types(mmc["allen_SUBC"])
mmc["allen_SUBC_incl_low_quality"] = mmc["allen_SUBC"].where(mmc.qc_passed, "Undefined")

adata = read_zarr(method_path / "sdata.zarr")["table"]
adata = adata[:, ~adata.var_names.str.startswith("Blank")]
adata.obsm["allen_cell_type_mapping"] = mmc.loc[adata.obs.index]
adata = anno_utils.process_adata(adata=adata, seg_method=seg_method, logger=logger)

leiden_col = f"leiden_res{leiden_res}".replace(".", "_")
if leiden_col not in adata.obs:
    sc.tl.leiden(adata, key_added=leiden_col, resolution=leiden_res)
labels = adata.obs["cell_type_mmc_incl_low_quality"]
vote = lambda cl: cl.map(labels.groupby(cl, observed=True).agg(lambda x: x.value_counts().idxmax())).astype(str)
adata.obs["cell_type_revised"] = vote(adata.obs[leiden_col])

sc.tl.leiden(adata, key_added="leiden_res20", resolution=20)
cl20 = adata.obs["leiden_res20"].astype(str)
adata.obs["cell_type_revised_res20"] = vote(cl20)

# pseudobulk MapMyCells: mean log2(1 + CPM) of QC-passed cells per res-20 cluster
import anndata as ad
from cellseg_benchmark.dea_utils import add_ensembl_id

passed = labels.ne("Undefined").to_numpy()
C = sp.csr_matrix(adata.layers["counts"][passed], dtype=np.float64)
log2cpm = (sp.diags(1e6 / np.ravel(C.sum(axis=1))) @ C).log1p() / np.log(2)
members = pd.get_dummies(cl20[passed]).astype(float)
centroids = sp.diags(1 / members.sum().to_numpy()) @ (sp.csr_matrix(members.T.to_numpy()) @ log2cpm)
pb = ad.AnnData(sp.csr_matrix(centroids), obs=pd.DataFrame(index=members.columns),
                var=pd.DataFrame(index=adata.var_names))
pb.var = add_ensembl_id(pb.var, species="mouse", out_col="ensmus_id", logger=logger)
anno_utils.run_mapmycells(pb, sample_name="pseudobulk", method_name=seg_method,
                          annotation_path=out_path, data_dir=data_dir, normalization="log2CPM")
pb_json = max((out_path / "mapmycells_out").glob(f"*MapMyCells_pseudobulk_{seg_method}.json"), key=os.path.getmtime)
with open(pb_json, "rb") as src:
    pbm = anno_utils.process_mapmycells_output(json.load(src))

frac_failed = pd.Series(~passed, index=cl20.index).groupby(cl20).mean()
pb_label = anno_utils.group_cell_types(pbm["allen_SUBC"]).reindex(frac_failed.index)
adata.obs["cell_type_revised_pb"] = cl20.map(pb_label.where(frac_failed <= 0.5).fillna("Undefined"))
print("pseudobulk correlation (SUBC):", pbm["allen_cor_SUBC"].describe().round(2).to_dict())

g = labels[passed].groupby(cl20[passed])
chk = pd.DataFrame({"per_cell_majority": g.agg(lambda x: x.value_counts().idxmax()),
                    "purity": g.agg(lambda x: x.value_counts(normalize=True).iloc[0]).round(2),
                    "pseudobulk": pb_label, "pb_cor": pbm["allen_cor_SUBC"].round(2)})
display(pd.crosstab(chk.per_cell_majority, chk.pseudobulk))

for col in [leiden_col, "leiden_res20"]:
    n = adata.obs[col].value_counts()
    print(f"{col}: {len(n)} clusters, median {n.median():.0f} cells")

## 3. Labels at each step

In [ ]:
steps = {
    "1. MMC per cell": adata.obs["cell_type_mmc_raw"].astype(str),
    "2. after SCALPEL QC": adata.obs["cell_type_mmc_incl_low_quality"].astype(str),
    "3. vote, Leiden res 10 (script)": adata.obs["cell_type_revised"],
    "3b. vote, Leiden res 20": adata.obs["cell_type_revised_res20"],
    "3c. pseudobulk MapMyCells, res 20": adata.obs["cell_type_revised_pb"],
}
old_csv = annotation_path / "adata_obs_annotated.csv"
if old_csv.exists() and "cell_id" in adata.obs:
    old = pd.read_csv(old_csv, usecols=["cell_id", "cell_type_revised"])
    old = old.set_index(old.cell_id.astype(str)).cell_type_revised
    steps["previous cell_type_revised"] = adata.obs.cell_id.astype(str).map(old).fillna("not in old csv").set_axis(adata.obs.index)

present = set().union(*[set(v.unique()) for v in steps.values()])
cats = [c for c in cell_type_colors if c in present] + sorted(present - set(cell_type_colors))
palette = {c: cell_type_colors.get(c, "#555555") for c in cats}

bases = ["umap"] + (["spatial"] if "spatial" in adata.obsm else [])
fig, axes = plt.subplots(len(steps), len(bases), figsize=(8 * len(bases), 6.5 * len(steps)), squeeze=False)
for r, (title, lab) in enumerate(steps.items()):
    adata.obs["_step"] = pd.Categorical(lab, categories=cats)
    for c, basis in enumerate(bases):
        sc.pl.embedding(adata, basis=basis, color="_step", palette=palette,
                        size=(220000 if basis == "umap" else 110000) / adata.n_obs,
                        legend_loc="on data" if basis == "umap" else None,
                        legend_fontsize=7, legend_fontoutline=1.5,
                        title=f"{title} ({basis})", ax=axes[r, c], show=False)
        axes[r, c].set_aspect("equal")
adata.obs.drop(columns="_step", inplace=True)
fig.legend([plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=palette[k], markersize=8) for k in cats],
           cats, loc="center right", fontsize=9, frameon=False)
plt.subplots_adjust(right=0.85, wspace=0.05, hspace=0.12)
plt.savefig(out_path / "3_labels_by_step.png", dpi=110, bbox_inches="tight")
plt.show()

counts = pd.DataFrame({k: v.value_counts() for k, v in steps.items()}).fillna(0).astype(int)
counts.sort_values(counts.columns[0], ascending=False)

## 4. Vascular markers

In [ ]:
markers = {"ECs": ["Cldn5", "Flt1", "Pecam1"], "Pericytes": ["Pdgfrb", "Kcnj8", "Vtn", "Rgs5"],
           "SMCs": ["Acta2", "Myh11"], "VLMCs": ["Dcn", "Col1a1"]}
markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}
markers = {k: v for k, v in markers.items() if v}
genes = sum(markers.values(), [])
print("in panel:", markers)

sc.pl.umap(adata, color=genes, layer="volume_log1p_norm", ncols=4, cmap="Reds",
           size=220000 / adata.n_obs, vmax="p99", show=False)
plt.savefig(out_path / "4_vascular_markers_umap.png", dpi=110, bbox_inches="tight")
plt.show()

for tag, name in [("pseudobulk", "3c. pseudobulk MapMyCells, res 20"), ("previous", "previous cell_type_revised")]:
    if name in steps:
        adata.obs["_grp"] = pd.Categorical(steps[name])
        sc.pl.dotplot(adata, markers, groupby="_grp", layer="volume_log1p_norm",
                      standard_scale="var", title=name, show=False)
        plt.savefig(out_path / f"4_vascular_markers_dotplot_{tag}.png", dpi=110, bbox_inches="tight")
        plt.show()
adata.obs.drop(columns="_grp", inplace=True, errors="ignore")

ec = sc.get.obs_df(adata, keys=markers["ECs"], layer="volume_log1p_norm").mean(axis=1)
pc = sc.get.obs_df(adata, keys=markers["Pericytes"], layer="volume_log1p_norm").mean(axis=1)
pd.DataFrame({name: {"ECs n": (v == "ECs").sum(), "ECs marker mean": ec[v == "ECs"].mean(),
                     "Pericytes n": (v == "Pericytes").sum(), "Pericytes marker mean": pc[v == "Pericytes"].mean(),
                     "Undefined %": 100 * (v == "Undefined").mean()}
              for name, v in steps.items() if not name.startswith(("1.", "2."))}).T.round(2)

### Pericyte island: what did MapMyCells call it?

Clusters with a high mean Kcnj8 + Vtn expression, their MapMyCells supertypes, correlation
and QC outcome. Many cells at correlation <= 0 in "0001 CLA-EPd-CTX Car3 Glut" means
MapMyCells could not map them and dumped them in the first supertype.

In [ ]:
expr = sc.get.obs_df(adata, keys=["Kcnj8", "Vtn"], layer="volume_log1p_norm").mean(axis=1)
score = expr.groupby(adata.obs[leiden_col], observed=True).mean()
peri = score.index[score > score.max() / 2]
in_peri = adata.obs[leiden_col].isin(peri)
print(f"{len(peri)} clusters, {in_peri.sum():,} cells")

display(pd.DataFrame({"n": adata.obs.loc[in_peri, leiden_col].astype(str).value_counts(), "Kcnj8+Vtn": score[peri].round(2)})
        .join(pd.DataFrame({k: v[in_peri].groupby(adata.obs.loc[in_peri, leiden_col], observed=True).first()
                            for k, v in steps.items() if not k.startswith(("1.", "2."))}))
        .sort_values("n", ascending=False))

m = mmc.loc[adata.obs.index[in_peri]]
print(f"correlation <= 0: {(m.allen_cor_SUPT <= 0).mean():.1%}   QC failed: {(~m.qc_passed).mean():.1%}")
display(m.groupby("allen_SUPT").agg(n=("qc_passed", "size"), qc_failed=("qc_passed", lambda x: (~x).sum()),
                                    median_cor=("allen_cor_SUPT", "median"))
        .sort_values("n", ascending=False).head(15))

## 5. Segmentation QC read-outs

In [ ]:
summary = pd.Series({
    "cells": adata.n_obs,
    "QC failed (per cell, %)": 100 * (adata.obs.cell_type_mmc_incl_low_quality == "Undefined").mean(),
    "median cor_SUPT": mmc.loc[adata.obs.index, "allen_cor_SUPT"].median(),
    "Undefined after vote (%)": 100 * (adata.obs.cell_type_revised == "Undefined").mean(),
}).round(3)
summary.to_csv(out_path / "4_summary.csv")
counts.to_csv(out_path / "3_counts_by_step.csv")
summary